# Shape of Beliefs — Replication

A pedagogical walkthrough of Sarfati et al., *The Shape of Beliefs: Geometry, Dynamics, and Interventions along Representation Manifolds of Language Models' Posteriors* (arXiv:2602.02315).

This notebook replicates the paper's core empirical findings: that Llama-3.2-1B, when shown number sequences sampled from a Gaussian, **encodes a posterior belief about that Gaussian's parameters as a curved manifold in its internal activations** — and that this manifold can be read out by linear probes.

**What you'll do here:** run Llama on sequences, cache its hidden states to Google Drive, train linear probes on those states, and reproduce Figures 1B and 3A from the paper.

**Who this is for:**
- **Future-you** revisiting the project — skim the TL;DR callouts, run the cells, done.
- **A peer researcher** replicating the paper — read the main narrative, trust the code, check the outputs.
- **A student learning mech-interp** — expand the "Deep dive" sections (collapsed by default), follow every step.

**Prerequisites:**
- Google Colab Pro (Pro+ recommended for longer sessions)
- A Google Drive with ~60GB free
- A Hugging Face account with access to Llama-3.2-1B approved ([request here](https://huggingface.co/meta-llama/Llama-3.2-1B))
- College-level linear algebra (vectors, matrices, PCA) and probability (Gaussians, cross-entropy) will help. We explain everything else.

**Time budget:** ~2 hours of wall-clock for the full pipeline, of which you'll be waiting for GPU for most of it. The setup is ~10 minutes of active work.

---

## What are we actually doing?

> **TL;DR** — Feed a language model random numbers drawn from a bell curve. Watch its internal state. Show that the model's brain encodes which bell curve it's seeing as a smooth, curved surface. Train simple linear classifiers to read that surface out.

### The setup in plain English

Imagine you show me this sequence of numbers:

```
533, 460, 689, 432, 501, 487, 508, 465, 340, ...
```

If you told me they were all sampled from the same distribution, I could eventually guess the distribution's shape: the numbers cluster around **500** (the mean, μ) and spread out about **100** in either direction (the standard deviation, σ). My brain is doing implicit Bayesian inference — updating a belief about (μ, σ) as I see more data.

The paper's central claim: **Llama-3.2-1B does the same thing, and we can watch it happen inside its neurons.** When Llama has seen enough numbers from N(μ=500, σ=100), its next-token prediction over the vocabulary of integers 0-999 looks like a Gaussian bell centered at 500 with width 100. Its internal "belief state" has converged.

### Why this is interesting

Language models are usually studied on messy problems: "does this model represent truth?", "is this sentence toxic?", "what is the model's opinion?". The answers are hard to pin down because the ground truth is fuzzy.

Gaussians are cleaner. You can *compute* what the correct posterior over (μ, σ) should be given the data. You can *vary* the generating parameters smoothly. And you can *read out* the model's belief directly from its logits. That makes this an ideal toy system for understanding how models encode and update beliefs.

### What the paper finds

1. **Geometry.** The model's internal states, when you vary μ, trace out a smooth **curved manifold** — not a straight line. Same for σ. So a "belief state" is not a single direction in activation space; it's a point on a 2D surface.

2. **Dynamics.** When the input distribution suddenly switches (e.g. from N(300, 100) to N(700, 100) mid-sequence), the model's internal state moves along the manifold, and the mean and variance equilibrate on *different* timescales.

3. **Interventions.** If you try to steer the model along a straight line from one belief to another, you push it *off* the manifold and get weird off-distribution outputs. Steering *along* the manifold preserves the belief family.

In this notebook we focus on **finding 1 (geometry)**. Findings 2 and 3 can be done in a follow-up notebook.

<details>
<summary><b>Deep dive: what is a "residual stream" and why do we probe it?</b></summary>

A transformer like Llama processes a sequence by passing information through a stack of layers. At each position in the sequence and at each layer, there's a vector of numbers — for Llama-3.2-1B, that vector has 2048 entries. This vector is called the **residual stream** or the **hidden state**. It's the model's "working memory" at that point.

Each layer reads the residual stream, computes something (attention + MLP), and adds the result back in. So the residual stream accumulates information as you go up the layers.

If the model has formed a belief about the input distribution, that belief has to be encoded *somewhere* in these 2048-dim vectors. **Probing** means training a simple classifier (usually linear) to extract a specific piece of information from the residual stream. If the classifier succeeds, the information is "linearly readable" at that layer.

Probes don't prove the model *uses* the information — only that it's present. But they're the first evidence we have that a model "knows" something internally.

</details>

<details>
<summary><b>Deep dive: why Gaussians? Why numbers?</b></summary>

Three reasons:

1. **Closed-form ground truth.** Given a Gaussian N(μ, σ), we know exactly what the correct next-token distribution over integers should be (a discretized Gaussian). We can compare the model's logits to this ground truth directly.

2. **Llama's tokenizer.** Llama-3 tokenizes every integer 0–999 as a *single token*. That means the model's next-number prediction is just a softmax over 1000 specific vocabulary entries — clean to extract, clean to interpret.

3. **Continuous parametrization.** μ and σ are real numbers. We can vary them continuously and see how the model's internal state moves. That's what lets us talk about "manifolds" — smoothly-varying surfaces — instead of just discrete categories.

Natural-language concepts (truth, sentiment, style) don't have this property. "Slightly more truthful" isn't a well-defined operation. "μ = 501 instead of μ = 500" is.

</details>

---
## Part 1: Setting up the environment

> **TL;DR** — Mount Drive, authenticate with Hugging Face, clone the repo, point the repo's data folder at Drive via a symlink. ~5 minutes.

We're going to run this pipeline across potentially multiple Colab sessions. Colab sessions are ephemeral — when yours disconnects, everything on the VM disk disappears. So we need to make sure our outputs land on **Google Drive**, which persists.

The trick we'll use: the repo's code writes outputs to a local `data/` folder. We'll **symlink** that folder to a location on Drive. The code doesn't know or care; the filesystem silently redirects writes to Drive. Zero code changes required.

### 1.1 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Sanity check: you should see some of your Drive folders
!ls /content/drive/MyDrive/ | head -5

MessageError: Error: credential propagation was unsuccessful

### 1.2 — Set up your Hugging Face token

Llama-3.2-1B is a **gated model**. To download it, you need:

1. A Hugging Face account
2. Approval for the specific model (go to the [Llama-3.2-1B page](https://huggingface.co/meta-llama/Llama-3.2-1B) and fill out Meta's form — usually instant approval)
3. A read-only access token from your [HF settings](https://huggingface.co/settings/tokens)

Once you have the token, store it in Colab's **Secrets manager** (the 🔑 key icon in the left sidebar):
- Name: `HF_TOKEN`
- Value: paste your token
- Toggle "Notebook access" on

Then the cell below loads it into the environment. This is safer than pasting the token into a cell — if you commit this notebook to GitHub, your token stays secret.

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print("Token loaded:", os.environ['HF_TOKEN'][:8] + "..." if os.environ.get('HF_TOKEN') else "MISSING")

### 1.3 — Clone the repo and install dependencies

The repo is at [sandguine/shape-of-beliefs](https://github.com/sandguine/shape-of-beliefs) — a fork of the Goodfire research team's replication code. If you're working from your own fork, change the URL below.

In [ ]:
!git clone https://github.com/sandguine/shape-of-beliefs.git
!cd shape-of-beliefs && uv pip install -e .

# Confirm the install worked
!cd shape-of-beliefs && python -c "import torch, transformers; print('torch:', torch.__version__); print('transformers:', transformers.__version__); print('CUDA available:', torch.cuda.is_available())"


**What to check:** the last line should print `CUDA available: True`. If it says False, you're on a CPU runtime. Go to **Runtime → Change runtime type** and pick a GPU (T4 is plenty; L4 or A100 is nicer but not needed).

### 1.4 — Symlink the data folder to Drive

This is the key setup step. After this cell, anything the pipeline writes to `shape-of-beliefs/data/` actually goes to `/content/drive/MyDrive/shape-of-beliefs-data/` on your Drive.

In [ ]:
!mkdir -p /content/drive/MyDrive/shape-of-beliefs-data
!rm -rf /content/shape-of-beliefs/data
!ln -s /content/drive/MyDrive/shape-of-beliefs-data /content/shape-of-beliefs/data

# Verify the symlink. You should see 'data -> /content/drive/MyDrive/shape-of-beliefs-data'
!ls -la /content/shape-of-beliefs/data

<details>
<summary><b>Deep dive: why symlinks instead of editing the code?</b></summary>

The authors' code hardcodes output paths. For example, `sequences_to_activations.py` writes to `<repo>/data/activations/...`. We have three ways to redirect this to Drive:

1. **Edit the code** to accept an `--output-dir` flag. Clean but means our fork drifts from upstream.
2. **Clone the whole repo onto Drive.** Works but Drive's filesystem is slower for reading Python source.
3. **Symlink** the `data/` folder only. Keeps source code on fast local disk, puts outputs on persistent Drive. Zero code changes.

Option 3 is the best tradeoff. The only catch is that Google Drive's FUSE filesystem can occasionally hiccup when writing many small files. The pipeline's built-in resume logic (it skips batches that already have output files) handles this gracefully — if a batch fails, just re-run and it picks up where it left off.

</details>

---
## Part 2: Stage 1 — Generating the input sequences

> **TL;DR** — Sample integers from N(μ, σ), write them to JSONL. Pure NumPy, fast, no GPU.

### What this stage does

We're going to feed Llama comma-separated strings of integers, like:

```
533,460,689,432,501,487,508,465,340,...
```

Each string is 1000 numbers long. Each batch of 10 strings shares the same generating distribution N(μ, σ). The paper sweeps over 17 different (μ, σ) combinations (9 values of μ at σ=100, and 9 values of σ at μ=500 — one is shared).

The script `generate_sequences.py` takes `--mean`, `--std`, `--num-seq`, `--len-seq`, draws the numbers, rounds and clips them to integers in [0, 999], and writes a JSONL file.

### 2.1 — First, a tiny smoke test

Before running the full pipeline, let's make sure everything is wired up. We'll generate just **2 sequences of 100 numbers each** from N(500, 100) — enough to prove the path works, cheap to throw away.

In [ ]:
!cd shape-of-beliefs && python generate_sequences.py --num-seq 2 --len-seq 100 --mean 500 --std 100

# Verify it landed on Drive via the symlink
!ls -la /content/drive/MyDrive/shape-of-beliefs-data/sequences/

You should see `gaussian_m500_s100_l100_n2.jsonl`. If not, the symlink isn't working — go back and re-run cell 1.4.

### 2.2 — Peek inside

Let's look at what we just generated:

In [ ]:
import json

path = '/content/drive/MyDrive/shape-of-beliefs-data/sequences/gaussian_m500_s100_l100_n2.jsonl'

with open(path) as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        print(f"Sequence {rec['sequence_id']}:")
        print(f"  First 200 chars: {rec['sequence_content'][:200]}...")
        print()

Each record has a `sequence_id` and a `sequence_content` string of comma-separated integers. Simple.

---
## Part 3: Stage 2 — Capturing Llama's activations

> **TL;DR** — Feed each sequence to Llama, hook into every layer, save the hidden states (and logits) to disk. Needs GPU. This is the expensive stage. **~45GB total** when running the full pipeline.

### What this stage does

For each sequence (a string of comma-separated numbers), we:

1. **Tokenize** it into a list of token IDs.
2. **Forward-pass** it through Llama-3.2-1B.
3. **Hook** every layer to grab the 2048-dim residual stream vector at every position.
4. **Restrict the logits** to just the 1000 number tokens (0-999) — the only ones we care about.
5. **Save** the hooked activations and restricted logits to `.pt` files.

The result is, for each (μ, σ) configuration, a folder of tensors on Drive.

<details>
<summary><b>Deep dive: what is a "hook" in PyTorch?</b></summary>

PyTorch lets you register a function that gets called every time a specific module runs its forward pass. That function receives the module's input and output tensors. You can use this to *save* internal states without modifying the model.

The paper's code does something like:

```python
activations = {}
def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

for i, layer in enumerate(model.model.layers):
    layer.register_forward_hook(make_hook(f"model_layers_{i}"))
```

After running the model, `activations` is a dict mapping layer names to their output tensors. That's how we get the residual stream at every layer without editing Llama's code.

</details>

<details>
<summary><b>Deep dive: why restrict to number tokens?</b></summary>

Llama's full vocabulary is ~128,000 tokens. If we saved the full logit distribution at every position for every layer, we'd be looking at terabytes.

But we only care about the probability Llama assigns to each integer 0–999. The repo includes a file `token_subset/llama3-2-1B_number_tokens.json` that lists the token IDs for those 1000 numbers. We use it to index into the logits and keep only those 1000 columns.

This reduces storage by 128x and loses nothing we need — the restricted softmax is exactly the next-integer distribution.

</details>

### 3.1 — Smoke test: run activations on the tiny sequence

If this works end-to-end, the full pipeline will work. This is the *real* test.

In [ ]:
# First run will download Llama-3.2-1B (~2GB). Takes ~1 min.
# Then it runs forward passes. Another ~1 min for this tiny input.
!cd shape-of-beliefs && python sequences_to_activations.py --dataset-name gaussian_m500_s100_l100_n2

In [ ]:
# Verify the outputs
!ls /content/drive/MyDrive/shape-of-beliefs-data/activations/gaussian_m500_s100_l100_n2/ | head -10
!ls /content/drive/MyDrive/shape-of-beliefs-data/logits/gaussian_m500_s100_l100_n2/ | head -10

You should see a bunch of `.pt` files, one per layer per batch. If the cell above failed with a 401 or 403 error, your HF token isn't working — revisit section 1.2. If it failed with an out-of-memory error, try **Runtime → Disconnect and delete runtime**, then reconnect with a bigger GPU.

### 3.2 — Sanity check: inspect one activation file

Let's load a single saved tensor and verify it has the shape we expect.

In [ ]:
import torch
import os

act_dir = '/content/drive/MyDrive/shape-of-beliefs-data/activations/gaussian_m500_s100_l100_n2/'
files = sorted(os.listdir(act_dir))
print("Activation files:", files[:5], "...")

# Pick one file and inspect it
sample = torch.load(os.path.join(act_dir, files[0]), map_location='cpu', weights_only=False)
print(f"\nFile: {files[0]}")
print(f"Type: {type(sample)}")
if hasattr(sample, 'shape'):
    print(f"Shape: {sample.shape}")
elif isinstance(sample, dict):
    for k, v in sample.items():
        if hasattr(v, 'shape'):
            print(f"  {k}: {v.shape}")
        else:
            print(f"  {k}: {type(v)}")

**What to expect:** the tensor should have shape `[batch_size, seq_len, hidden_dim]` where `batch_size` is 2 (two sequences), `seq_len` is roughly 200 (100 numbers × 2 tokens per number: the number and the comma), and `hidden_dim` is 2048.

Actual layout may differ slightly depending on how the script packs the data — don't worry about the exact numbers, just that the shape makes sense.

---
## Part 4: Running the full pipeline

> **TL;DR** — Run `scripts/generate_all.sh`. This kicks off all 17 datasets. Takes ~1–2 hours. Produces ~45GB. Resume-safe.

### What this will do

The shell script `generate_all.sh` loops over 17 (μ, σ) configurations and runs stages 1 and 2 for each. Specifically:

- **μ sweep** at σ=100: μ ∈ {300, 350, 400, 450, 500, 550, 600, 650, 700}
- **σ sweep** at μ=500: σ ∈ {10, 20, 30, 50, 80, 100, 120, 150, 200}
- Plus one **combined** dataset that concatenates m=300 and m=700 sequences — used in the paper's dynamics analysis.

Each dataset has 10 sequences of 1000 numbers. Total: ~170K numbers going through Llama at 16 layers each, cached to disk.

### If Colab disconnects mid-run

That's fine. The pipeline skips any batch that already has an output file. Just re-run the cell and it resumes. In practice, for a T4 runtime you may want to babysit the session occasionally to keep it alive.

### 4.1 — Run the full pipeline

In [ ]:
# This will take 1–2 hours. Output will scroll for a while.
# Progress: 17 datasets × 10 sequences × 16 layers of activations each.
!cd shape-of-beliefs && chmod +x scripts/generate_all.sh && ./scripts/generate_all.sh

### 4.2 — Verify the full output

In [ ]:
import os

base = '/content/drive/MyDrive/shape-of-beliefs-data'

# How many datasets have activations?
act_datasets = os.listdir(f"{base}/activations")
print(f"Activation datasets: {len(act_datasets)}")
for ds in sorted(act_datasets):
    n_files = len(os.listdir(f"{base}/activations/{ds}"))
    print(f"  {ds}: {n_files} files")

# Total size on Drive
!du -sh /content/drive/MyDrive/shape-of-beliefs-data/

Expect to see 17 single datasets + 1 combined dataset, with roughly the same number of files per dataset, totaling around 45GB. If something is missing, just re-run cell 4.1 and it'll fill in the gaps.

---
## Part 5: Stage 3 — Training linear field probes

> **TL;DR** — Train a small linear classifier that predicts μ from the cached activations. If it works, Llama's belief about μ is linearly decodable. This runs on CPU and is fast.

### What a linear probe is, mathematically

Given a set of activation vectors $x \in \mathbb{R}^{2048}$ labeled with which μ value they came from (one of {300, 350, ..., 700}), we train a multiclass classifier:

$$p(\mu = c \mid x) = \frac{\exp(w_c^\top x)}{\sum_{c'} \exp(w_{c'}^\top x)}$$

This is standard softmax regression. Each class $c$ gets its own weight vector $w_c \in \mathbb{R}^{2048}$. We train by minimizing cross-entropy.

If the test accuracy is high, it means **different μ values produce linearly separable activations**. That's the first sign that μ is represented in a structured way inside the model.

### Why "field" probes?

A regular probe would separate {cats, dogs, horses} — three unrelated categories. A **linear field probe** separates categories that have a continuous ordering, like μ = 300 vs 350 vs 400 vs ... 700. The probe vectors $w_c$ vary smoothly with $c$, tiling the manifold of activations.

We'll verify this smooth variation in the next section by looking at the cosine similarity matrix of the $w_c$ vectors.

<details>
<summary><b>Deep dive: why is this called "field" probing?</b></summary>

In physics, a **field** assigns a value to every point in a space — like temperature at every point in a room, or gravitational force at every point in the solar system. Fields have structure: nearby points tend to have similar values.

A "linear field probe" generalizes a linear probe from a **single direction** (one weight vector) to a **family** of directions, one per value of the continuous parameter. The probe vectors vary smoothly with the parameter, like a field of vectors indexed by μ.

Yocum et al. (2025) introduced this terminology explicitly. The Shape of Beliefs paper adopts it to describe how μ is encoded across the activation manifold.

</details>

### 5.1 — Run the probe training

The script trains a probe at a specific layer. The paper shows probes work well at all layers ≥ 5 or so. Let's start with **layer 14** (late in the network, where belief representations should be most refined).

In [ ]:
# Train probes at layer 14. Takes ~5 min on CPU.
!cd shape-of-beliefs && python linear_field_probes.py --layer 14 --epochs 100

### 5.2 — Load the trained probe and inspect

In [ ]:
import torch

probe_path = '/content/shape-of-beliefs/probes/epoch100_biasFalse/linear_probe_layer14.pt'
probe = torch.load(probe_path, map_location='cpu', weights_only=False)

print("Probe checkpoint keys:", list(probe.keys()))
print("\nTest accuracy:", probe.get('test_accuracy', '(not found)'))
print("Probe weight shape:", probe['probe_state_dict']['linear.weight'].shape if 'probe_state_dict' in probe else '(not found)')

**What to expect:** test accuracy should be very high (> 0.95). The weight matrix should be shape `[num_classes, 2048]` — one row per μ value.

---
## Part 6: Figure 1B — The belief manifold

> **TL;DR** — Average activations per (μ, σ), project to 3D via PCA, plot. You'll see two perpendicular curved "strings of beads" — one for varying μ, one for varying σ.

### What we're plotting

For each of the 17 datasets, we compute the **mean activation** across all positions and all sequences at a chosen layer (layer 14 again). That gives us 17 vectors in $\mathbb{R}^{2048}$.

We then project these 17 points down to 3D using **PCA** (principal component analysis). PCA finds the 3 orthogonal directions along which the points vary the most.

If μ and σ are encoded as smooth, curved paths in activation space, the projection should show two visible curves — a "μ curve" where μ varies and σ is fixed at 100, and a "σ curve" where σ varies and μ is fixed at 500. The two curves should meet at the point (μ=500, σ=100).

<details>
<summary><b>Deep dive: what is PCA, briefly?</b></summary>

Given N points in a high-dimensional space, PCA finds the orthogonal axes that capture the most variance:

- The **first principal component** is the direction along which the points are most spread out.
- The **second** is the direction orthogonal to the first that captures the most remaining variance.
- And so on.

Projecting onto the top K principal components gives you a K-dimensional view of your data that preserves as much of the spread as possible. If K=2 or 3, you can plot it.

Crucially, PCA is **linear** — it doesn't create curves. If the PCA projection *looks* curved, that's because the underlying data actually lies on a curved surface; PCA is just showing you a flat projection of that curvature.

</details>

### 6.1 — Compute mean activations per dataset

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path

LAYER = 14
BASE = '/content/drive/MyDrive/shape-of-beliefs-data'

# The 17 datasets from the paper
mu_datasets = [f"gaussian_m{m}_s100_l1000_n10" for m in [300, 350, 400, 450, 500, 550, 600, 650, 700]]
sigma_datasets = [f"gaussian_m500_s{s:03d}_l1000_n10" for s in [10, 20, 30, 50, 80, 100, 120, 150, 200]]
all_datasets = list(dict.fromkeys(mu_datasets + sigma_datasets))  # dedupe (m500_s100 appears in both)

print(f"Processing {len(all_datasets)} datasets at layer {LAYER}")

mean_activations = {}
for ds in all_datasets:
    ds_dir = Path(BASE) / 'activations' / ds
    if not ds_dir.exists():
        print(f"  SKIP {ds} (not found)")
        continue
    # Load all batches for this layer, average across sequences and positions
    files = sorted(ds_dir.glob(f'model_layers_{LAYER}_batch*.pt'))
    if not files:
        print(f"  SKIP {ds} (no layer {LAYER} files)")
        continue
    batches = [torch.load(f, map_location='cpu', weights_only=False) for f in files]
    # Each batch tensor is roughly [batch, seq_len, hidden_dim]
    # We average over batch and seq_len to get one vector per dataset
    stacked = torch.cat([b.reshape(-1, b.shape[-1]) for b in batches], dim=0)
    mean_activations[ds] = stacked.mean(dim=0).numpy()
    print(f"  OK   {ds}: mean shape {mean_activations[ds].shape}")


### 6.2 — Project to 3D with PCA and plot

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: registers 3D projection

# Stack all mean activations into a matrix
names = list(mean_activations.keys())
X = np.stack([mean_activations[n] for n in names])  # [17, 2048]

# PCA to 3D
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)  # [17, 3]

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance captured by first 3 PCs: {pca.explained_variance_ratio_.sum():.3f}")

# Split into mu-curve (varying mu, sigma=100) and sigma-curve (mu=500, varying sigma)
def get_mu(ds):
    return int(ds.split('_m')[1].split('_')[0])

def get_sigma(ds):
    return int(ds.split('_s')[1].split('_')[0])

mu_points = [(get_mu(n), X_pca[i]) for i, n in enumerate(names) if get_sigma(n) == 100]
sigma_points = [(get_sigma(n), X_pca[i]) for i, n in enumerate(names) if get_mu(n) == 500]

mu_points.sort()
sigma_points.sort()

# Plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Mu curve: pink to green gradient
mu_values = [p[0] for p in mu_points]
mu_coords = np.array([p[1] for p in mu_points])
mu_colors = plt.cm.PiYG(np.linspace(0.1, 0.9, len(mu_points)))
ax.plot(mu_coords[:, 0], mu_coords[:, 1], mu_coords[:, 2], '-', color='gray', alpha=0.5)
for i, (mu, coord) in enumerate(mu_points):
    ax.scatter(*coord, color=mu_colors[i], s=80, label=f'μ={mu}' if i % 2 == 0 else None)

# Sigma curve: shades of purple
sigma_values = [p[0] for p in sigma_points]
sigma_coords = np.array([p[1] for p in sigma_points])
sigma_colors = plt.cm.Purples(np.linspace(0.3, 0.9, len(sigma_points)))
ax.plot(sigma_coords[:, 0], sigma_coords[:, 1], sigma_coords[:, 2], '-', color='gray', alpha=0.5)
for i, (sigma, coord) in enumerate(sigma_points):
    ax.scatter(*coord, color=sigma_colors[i], s=80, marker='s')

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title(f'Belief manifolds at layer {LAYER}\nCircles: varying μ (σ=100).  Squares: varying σ (μ=500).')
plt.tight_layout()
plt.show()


**What to expect:** two visible curves in 3D space that look like *strings of beads*. The μ-curve (circles, pink-to-green) should form a smooth arc as μ goes from 300 to 700. The σ-curve (squares, purple shades) should form a perpendicular-ish arc as σ varies.

Compare to Figure 1B in the paper. The general shape should match; exact orientation will differ (PCA axes are arbitrary up to sign).

**If the plot looks like random dots:** something is off. Most likely the layer choice — try layer 10 or 8. Very late layers can be dominated by next-token prediction and lose some of the geometry.

---
## Part 7: Figure 3A — Probe separability across layers

> **TL;DR** — Train a probe at every layer. Plot test accuracy vs layer. Shows where in the network the belief representation becomes linearly decodable.

### What we're plotting

Llama-3.2-1B has 16 layers (plus the embedding, so call it layers 0-15 or 0-16 depending on how you count). At every layer, we train a fresh probe on the μ-sweep activations. We plot test accuracy as a function of layer.

The paper shows this is ~0.87 at layer 0 and climbs to ~0.99 by layer 15. The interesting qualitative story is **where** it rises.

### 7.1 — Train probes at every layer

This takes a while (~5 min × 16 layers = ~80 min). If you want to sample, uncomment the smaller list.

In [ ]:
import subprocess

layers_to_probe = list(range(0, 16))  # all layers
# layers_to_probe = [0, 3, 6, 9, 12, 15]  # smaller sample if you're short on time

for layer in layers_to_probe:
    probe_file = f'/content/shape-of-beliefs/probes/epoch100_biasFalse/linear_probe_layer{layer}.pt'
    if os.path.exists(probe_file):
        print(f"Layer {layer}: already trained, skipping")
        continue
    print(f"Layer {layer}: training...")
    result = subprocess.run(
        ['python', 'linear_field_probes.py', '--layer', str(layer), '--epochs', '100'],
        cwd='/content/shape-of-beliefs',
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  FAILED: {result.stderr[-500:]}")
    else:
        print(f"  done")


### 7.2 — Load all probes and plot accuracy vs layer

In [ ]:
import torch
import matplotlib.pyplot as plt

accuracies = {}
for layer in layers_to_probe:
    probe_file = f'/content/shape-of-beliefs/probes/epoch100_biasFalse/linear_probe_layer{layer}.pt'
    if not os.path.exists(probe_file):
        continue
    probe = torch.load(probe_file, map_location='cpu', weights_only=False)
    # The key may be 'test_accuracy' or similar — inspect if missing
    acc = probe.get('test_accuracy') or probe.get('accuracy') or probe.get('test_acc')
    if acc is not None:
        accuracies[layer] = float(acc)

layers = sorted(accuracies.keys())
accs = [accuracies[l] for l in layers]

plt.figure(figsize=(8, 5))
plt.plot(layers, accs, 'o-', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('Layer')
plt.ylabel('Test accuracy')
plt.title('Linear probe separability of μ across layers')
plt.ylim(0.8, 1.02)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAccuracies by layer:")
for l in layers:
    print(f"  Layer {l}: {accuracies[l]:.3f}")


**What to expect:** accuracy starts fairly high (~0.85) even at layer 0 (because the number tokens are pretty distinguishable just from embedding), climbs steadily, and plateaus near 1.0 by the later layers.

Compare to Figure 3A. If you see the same general shape — monotonically increasing, plateauing at the top — you've reproduced the paper's core claim that μ is a linearly decodable feature across all layers.

---
## Conclusion: what you just did, and where to go from here

You:

1. Fed Llama-3.2-1B thousands of numbers sampled from Gaussians with varying (μ, σ).
2. Cached its internal activations and next-token predictions to Google Drive.
3. Confirmed via 3D projection that different belief states (μ, σ) live on smooth curved manifolds in Llama's activation space.
4. Confirmed via linear probes that μ is linearly readable from activations at every layer, with accuracy climbing from ~0.85 at layer 0 to ~0.99 at late layers.

Together, these are the evidence for the paper's **Geometry** claim: that posterior beliefs are encoded as structured, continuous objects — not as isolated linear directions.

### What we didn't cover

The paper also investigates two more questions, which could be their own follow-up notebooks:

- **Dynamics (Section 3.2):** when the generating distribution suddenly changes, how does the model's belief trajectory move through activation space? See the `gaussian_m300_s100_l1000_n10+gaussian_m700_s100_l1000_n10` combined dataset (already in your cached data) for the raw material.
- **Interventions (Section 4):** compare linear steering vs manifold-aware steering vs field-aware steering. The Streamlit app (`app/steering_explorer_app.py` in the repo) is the authors' interactive exploration.

### If you're learning

Some worthwhile next steps:

- Implement one of the baseline probes manually (use `torch.nn.Linear(2048, 9)` and train it yourself on cached activations). Match the accuracy and make sure you understand every step.
- Replot Figure 1B at a **different layer** and compare. The manifold geometry changes with depth.
- Compute the cosine similarity matrix of the probe weight vectors (available in the saved probe checkpoint). Plot it as a heatmap. Smooth off-diagonal decay is the signature of a "field" probe.

### If you're replicating for research

- Check that your figures visually match the paper's. If they don't, the most likely culprits are (a) wrong layer, (b) wrong aggregation (positional averaging vs last-token only — the paper specifies "com2num tokens"), or (c) PCA sign/rotation artifacts.
- Cross-reference the notebooks in `figures/` in the repo — those are the authors' exact plotting code.

### Citation

If you use this replication in your own work, please cite the original paper:

```bibtex
@article{Sarfati2026ShapeOfBeliefs,
  title={The Shape of Beliefs: Geometry, Dynamics, and Interventions along Representation Manifolds of Language Models' Posteriors},
  author={Sarfati, Rapha\"el and Bigelow, Eric and Wurgaft, Daniel and Merullo, Jack and Geiger, Atticus and Lewis, Owen and McGrath, Tom and Singh Lubana, Ekdeep},
  journal={arXiv preprint arXiv:2602.02315},
  year={2026}
}
```

---

*This notebook is a replication draft. See [sandguine/shape-of-beliefs](https://github.com/sandguine/shape-of-beliefs) for the code.*